# Mapping Unique Olist Customers To Texas ZIPs

This notebook assigns one Texas ZIP to each `customer_unique_id`, then joins that assignment back to every original customer-order row.


In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "rebuild_unique_customer_mapping.py").exists():
            return candidate
    raise FileNotFoundError("Could not find rebuild_unique_customer_mapping.py from the current notebook path.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.options.display.max_columns = 120

from rebuild_unique_customer_mapping import (
    POSTAL_CUSTOMER_ORDERS_CSV,
    POSTAL_CUSTOMERS_CSV,
    SOURCE_CODE_SUMMARY_CSV,
    SOURCE_CODE_TO_TEXAS_ZIP_MAPPING_CSV,
    TEXAS_TARGET_WEIGHTS_CSV,
    build_final_outputs,
    build_mapping_table,
    load_olist_customers,
)

from rebuild_unique_customer_mapping import OLIST_UNIQUE_CUSTOMERS_WITH_SOURCE_CSV


In [ ]:
unique_customers_df = pd.read_csv(
    OLIST_UNIQUE_CUSTOMERS_WITH_SOURCE_CSV,
    dtype={
        "canonical_customer_zip_code_prefix": "string",
        "zip_group_4": "string",
        "zip_group_3": "string",
        "zip_group_2": "string",
        "source_code": "string",
    },
)
source_code_summary_df = pd.read_csv(SOURCE_CODE_SUMMARY_CSV, dtype={"source_code": "string"})
texas_target_weights_df = pd.read_csv(TEXAS_TARGET_WEIGHTS_CSV, dtype={"zip": "string", "county_geoid": "string"})
olist_customers_df = load_olist_customers()

texas_target_weights_df["zip"] = texas_target_weights_df["zip"].str.zfill(5)

print(f"Unique customer rows: {len(unique_customers_df):,}")
print(f"Source-code rows: {len(source_code_summary_df):,}")
print(f"Texas target ZIP rows: {len(texas_target_weights_df):,}")
print(f"Original customer-order rows: {len(olist_customers_df):,}")


In [ ]:
mapping_table_df = build_mapping_table(source_code_summary_df, texas_target_weights_df)

print(f"Mapped source codes: {len(mapping_table_df):,}")
print(f"Unique assigned Texas ZIPs: {mapping_table_df['assigned_texas_zip'].nunique():,}")
display(mapping_table_df.head(20))


In [ ]:
postal_customers_df, postal_customer_orders_df = build_final_outputs(
    unique_customers_df,
    olist_customers_df,
    mapping_table_df,
)

print(f"postal_customers rows: {len(postal_customers_df):,}")
print(f"postal_customer_orders rows: {len(postal_customer_orders_df):,}")
display(postal_customers_df.head())
display(postal_customer_orders_df.head())


In [ ]:
mapping_table_df.to_csv(SOURCE_CODE_TO_TEXAS_ZIP_MAPPING_CSV, index=False)
postal_customers_df.to_csv(POSTAL_CUSTOMERS_CSV, index=False)
postal_customer_orders_df.to_csv(POSTAL_CUSTOMER_ORDERS_CSV, index=False)

print(f"Wrote source-code to Texas ZIP mapping: {SOURCE_CODE_TO_TEXAS_ZIP_MAPPING_CSV}")
print(f"Wrote customer-level postal output: {POSTAL_CUSTOMERS_CSV}")
print(f"Wrote order-level postal output: {POSTAL_CUSTOMER_ORDERS_CSV}")


In [ ]:
split_zip_customers = (
    postal_customer_orders_df.groupby("customer_unique_id")["assigned_texas_zip"].nunique(dropna=False) > 1
).sum()

validation = pd.DataFrame(
    {
        "check": [
            "postal_customers rows",
            "postal_customers unique customer_unique_id",
            "postal_customer_orders rows",
            "raw Olist customer-order rows",
            "customers with multiple assigned Texas ZIPs",
            "missing customer-level ZIP assignments",
            "missing order-level ZIP assignments",
        ],
        "value": [
            len(postal_customers_df),
            postal_customers_df["customer_unique_id"].nunique(),
            len(postal_customer_orders_df),
            len(olist_customers_df),
            int(split_zip_customers),
            int(postal_customers_df["assigned_texas_zip"].isna().sum()),
            int(postal_customer_orders_df["assigned_texas_zip"].isna().sum()),
        ],
    }
)

display(validation)
